In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

# Ayarlar
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Senin verinin bulunduğu klasör yolu
INFANT_DATA_PATH = r"C:\Users\Ayberk\Desktop\Ayberk Dosyalar\Bitirme Projesi\Fundus\all_fundus_datas\vessel_segment_bebek_data"

# Kaydettiğin en iyi modelin adı
MODEL_PATH = "best_bebek_infant_model2.pth"


In [ ]:
class InfantFundusDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.img_dir = os.path.join(root_dir, split, "image")
        self.mask_dir = os.path.join(root_dir, split, "mask")
        
        valid_extensions = ('.png', '.jpg', '.jpeg', '.tif', '.bmp')
        self.image_names = [f for f in os.listdir(self.img_dir) 
                            if f.lower().endswith(valid_extensions)]
        
        self.transform = transform
        print(f"--> {split} seti için {len(self.image_names)} adet görsel bulundu.")

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Dosya adını uzantısız al (örn: "017_fundus")
        base_name = os.path.splitext(img_name)[0]
        
        # Maskeyi bulmak için farklı uzantıları dene
        mask_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.tif', '.png']:
            temp_path = os.path.join(self.mask_dir, base_name + ext)
            if os.path.exists(temp_path):
                mask_path = temp_path
                break
        
        # Görüntü ve Maskeyi oku
        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) if mask_path else None

        if image is None:
            raise FileNotFoundError(f"Görüntü okunamadı: {img_path}")
        if mask is None:
            raise FileNotFoundError(f"Maske bulunamadı veya okunamadı! Aranan isim: {base_name} (Klasör: {self.mask_dir})")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask

In [9]:
# 1. Mimariyi Tanımla (3 Sınıf: BG, Arter, Ven)
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b5",
    encoder_weights=None, 
    in_channels=3,
    classes=3, 
).to(DEVICE)


# 3. Test Transform ve Dataset
test_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

test_ds = InfantFundusDataset(INFANT_DATA_PATH, split="val", transform=test_transform)


--> val seti için 12 adet görsel bulundu.


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp

# 1. Renk Düzeltme (Denormalizasyon)
# Orijinal fundus renklerini geri getirir 
def denormalize(img_tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = (img * std) + mean 
    return np.clip(img, 0, 1)

# 2. Bebek Datası Renk Haritası (0: Arka Plan, 1: Arter, 2: Ven) [cite: 86, 193]
infant_colors = {0: [0,0,0], 1: [255,0,0], 2: [0,0,255]}

def infant_mask_to_rgb(mask):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in infant_colors.items():
        rgb[mask == cls] = color
    return rgb

# 3. Model Yükleme
model.load_state_dict(torch.load("best_bebek_infant_model2.pth"))
model.eval()

print(f"--- Bebek Datası Analizi (İlk 10 Örnek) ---")

with torch.no_grad():
    for i in range(min(10, len(test_ds))):
        image, mask = test_ds[i]
        input_tensor = image.unsqueeze(0).to(DEVICE)
        
        output = model(input_tensor)
        pred = torch.argmax(output, dim=1).squeeze(0)
        
        # Metrik İstatistikleri [cite: 178]
        # num_classes=3 (BG, Arter, Ven)
        tp, fp, fn, tn = smp.metrics.get_stats(
            pred.unsqueeze(0), # Batch boyutu ekle: [1, H, W]
            mask.unsqueeze(0).to(DEVICE).long(), 
            mode='multiclass', 
            num_classes=3
        )
        
        # HATA DÜZELTMESİ: reduction="micro" kullanarak boyut hatasını önlüyoruz
        # Artery (Sınıf 1) ve Vein (Sınıf 2) Dice Skorları [cite: 188]
        artery_dice = smp.metrics.f1_score(tp[:, 1], fp[:, 1], fn[:, 1], tn[:, 1], reduction="micro").item()
        vein_dice = smp.metrics.f1_score(tp[:, 2], fp[:, 2], fn[:, 2], tn[:, 2], reduction="micro").item()

        # Panel Oluşturma
        plt.figure(figsize=(16, 5))
        
        # Orijinal (Normalizasyon geri alınmış )
        plt.subplot(1, 3, 1)
        plt.title(f"Bebek Fundus: {test_ds.image_names[i]}")
        plt.imshow(denormalize(image))
        plt.axis('off')

        # Gerçek Etiket [cite: 148]
        plt.subplot(1, 3, 2)
        plt.title("Uzman Etiketi (GT)")
        plt.imshow(infant_mask_to_rgb(mask.numpy() if torch.is_tensor(mask) else mask))
        plt.axis('off')

        # Model Tahmini ve Skorlar 
        plt.subplot(1, 3, 3)
        plt.title(f"Tahmin\nArter Dice: {artery_dice:.4f} | Ven Dice: {vein_dice:.4f}")
        plt.imshow(infant_mask_to_rgb(pred.cpu().numpy()))
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()